# Calling an A2A Agent using an A2A Client

In this lesson, you will use the A2A Python Client to interact with the Policy Agent you created and activated in the previous lesson. The client handles the protocol details, allowing you to discover the agent's capabilities via its `AgentCard` and send messages to it.

In [ ]:
import httpx
from IPython.display import Markdown, display
from a2a.client import (
    Client,
    ClientConfig,
    ClientFactory,
    create_text_message_object,
)
from a2a.types import AgentCard, Artifact, Message, Task
from a2a.utils.message import get_message_text


## Setup Client Configuration

In [ ]:
host = "localhost"
port = 9999

In [ ]:
prompt = "How much would I pay for mental health therapy?"

## Visualize Agent Card

This helper function is designed to nicely format and display the `AgentCard` data returned by the server, helping you verify that the agent is discoverable and configured correctly.


In [ ]:
def display_agent_card(agent_card: AgentCard) -> None:
    def esc(text: str) -> str:
        """Escapes pipe characters for Markdown table compatibility."""
        return str(text).replace("|", r"\|")

    # --- Part 1: Main Metadata Table ---
    md_parts = [
        "### Agent Card Details",
        "| Property | Value |",
        "| :--- | :--- |",
        f"| **Name** | {esc(agent_card.name)} |",
        f"| **Description** | {esc(agent_card.description)} |",
        f"| **Version** | `{esc(agent_card.version)}` |",
        f"| **URL** | [{esc(agent_card.url)}]({agent_card.url}) |",
        f"| **Protocol Version** | `{esc(agent_card.protocol_version)}` |",
    ]

    # --- Part 2: Skills Table ---
    if agent_card.skills:
        md_parts.extend(
            [
                "\n#### Skills",
                "| Name | Description | Examples |",
                "| :--- | :--- | :--- |",
            ]
        )
        for skill in agent_card.skills:
            examples_str = (
                "<br>".join(f"• {esc(ex)}" for ex in skill.examples)
                if skill.examples
                else "N/A"
            )
            md_parts.append(
                f"| **{esc(skill.name)}** | {esc(skill.description)} | {examples_str} |"
            )

    # Join all parts and display
    display(Markdown("\n".join(md_parts)))

## Run the Client Interaction

Now you will execute the client interaction flow:
1.  **Connect**: Establish a connection to the agent using `ClientFactory`.
2.  **Discover**: Retrieve the `AgentCard` using `get_card()`.
3.  **Construct**: Create a message object with the user's prompt.
4.  **Send & Receive**: Send the message and asynchronously process the response stream (which may include Tasks, Artifacts, or Messages).


In [ ]:
async with httpx.AsyncClient(timeout=100.0) as httpx_client:
    # Step 1: Create a client
    client: Client = await ClientFactory.connect(
        f"http://{host}:{port}",
        client_config=ClientConfig(
            httpx_client=httpx_client,
        ),
    )

    # Step 2: Discover the agent by fetching its card
    agent_card = await client.get_card()
    display_agent_card(agent_card)

    # Step 3: Create the message using a convenient helper function
    message = create_text_message_object(content=prompt)

    display(Markdown(f"**Sending prompt:** `{prompt}` to the agent..."))

    # Step 4: Send the message and await the final response.
    responses = client.send_message(message)

    text_content = ""

    # Step 5: Process the responses from the agent
    async for response in responses:
        if isinstance(response, Message):
            # The agent replied directly with a final message
            print(f"Message ID: {response.message_id}")
            text_content = get_message_text(response)
        # response is a ClientEvent
        elif isinstance(response, tuple):
            task: Task = response[0]
            print(f"Task ID: {task.id}")
            if task.artifacts:
                artifact: Artifact = task.artifacts[0]
                print(f"Artifact ID: {artifact.artifact_id}")
                text_content = get_message_text(artifact)

    display(Markdown("### Final Agent Response\n-----"))
    if text_content:
        display(Markdown(text_content))
    else:
        display(
            Markdown(
                """**No final text content received or task did not 
                complete successfully.**"""
            )
        )

## Resources

- [A2A Python SDK Documentation](https://a2a-protocol.org/latest/sdk/python/api/)
